<h3 style="color:#6FA8DC; font-weight:bold">05 — One-Hot Encoding</h3>

<h5 style="color:#78B89A; font-weight:bold;">OneHotEncoder, Dummy Variable Trap, Most Frequent Categories and Practical Example</h5>

In the previous notebook, we studied Ordinal Encoding and Label Encoding.

Now we will study **One-Hot Encoding**, one of the most commonly used techniques for converting categorical input features into numerical form.

This notebook covers:

- What is One-Hot Encoding?
- Why do we need it?
- Complete transformation process
- One-Hot Encoding using pandas
- One-Hot Encoding using Scikit-learn
- Dummy variables
- Dummy Variable Trap
- Why one encoded column is removed
- `drop='first'`
- Handling unknown categories
- Encoding the most frequent category
- Practical example using the customer dataset
- Common mistakes and final revision

<div style="border-top:black 2px solid"></div>

# 1. What is One-Hot Encoding?

One-Hot Encoding converts categorical values into separate binary columns.

Each category gets its own column.

The value is:

- `1` if the row belongs to that category.
- `0` if the row does not belong to that category.

### Example

Suppose we have one column:

| City |
|---|
| Delhi |
| Mumbai |
| Pune |
| Delhi |

After One-Hot Encoding:

| City_Delhi | City_Mumbai | City_Pune |
|---:|---:|---:|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |
| 1 | 0 | 0 |

This is called **one-hot representation** because exactly one category column is active for each row.

# 2. Why do we need One-Hot Encoding?

Machine Learning algorithms generally require numerical inputs.

Suppose we encode cities like this:

```text
Delhi  → 0
Mumbai → 1
Pune   → 2
```

This can create an artificial relationship:

```text
Pune > Mumbai > Delhi
```

But cities do not have a natural ranking.

One-Hot Encoding avoids this problem by creating independent binary columns.

### One-Hot Encoding is commonly used for:

- City
- Gender
- Browser
- Payment Method
- Department
- Colour
- Country
- Product Category
- Device Type

These are usually **nominal categorical variables**.

<div style="border-top:black 2px solid"></div>

# 3. Complete Process: Column → One-Hot Columns → Dummy Variables

This is the complete transformation process.

## Step 1 — Original categorical column

```text
City
----
Delhi
Mumbai
Pune
Delhi
```

## Step 2 — Create one column for each category

```text
City_Delhi
City_Mumbai
City_Pune
```

## Step 3 — Fill the columns with 0 and 1

| City | City_Delhi | City_Mumbai | City_Pune |
|---|---:|---:|---:|
| Delhi | 1 | 0 | 0 |
| Mumbai | 0 | 1 | 0 |
| Pune | 0 | 0 | 1 |
| Delhi | 1 | 0 | 0 |

These new binary columns are called **dummy variables**.

## Step 4 — Remove one dummy column when required

For three categories, we can keep only two columns:

| City | City_Mumbai | City_Pune |
|---|---:|---:|
| Delhi | 0 | 0 |
| Mumbai | 1 | 0 |
| Pune | 0 | 1 |
| Delhi | 0 | 0 |

Here, Delhi becomes the reference or baseline category.

This helps avoid perfect multicollinearity in models that use an intercept, such as Linear Regression and Logistic Regression.

# 4. What are Dummy Variables?

A dummy variable is a binary numerical variable that indicates whether an observation belongs to a particular category.

It usually contains only:

```text
0 or 1
```

Example:

```text
Gender_Male
```

| Gender | Gender_Male |
|---|---:|
| Male | 1 |
| Female | 0 |
| Male | 1 |
| Female | 0 |

### Interpretation

- `1` → The person is male.
- `0` → The person is not male.

When a categorical column is converted into multiple binary columns, those columns are called dummy variables.

<div style="border-top:black 2px solid"></div>

# 5. One-Hot Encoding using Pandas

Pandas provides:

```python
pd.get_dummies()
```

This is a quick and convenient way to encode categorical columns.

### Basic syntax

```python
pd.get_dummies(
    data,
    columns=['column_name'],
    dtype=int
)
```

In [ ]:
import pandas as pd
import numpy as np

city_df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Pune', 'Delhi']
})

city_df

In [ ]:
pd.get_dummies(
    city_df,
    columns=['City'],
    dtype=int
)

### Understanding the output

Pandas:

1. Takes the original categorical column.
2. Finds its unique categories.
3. Creates one column per category.
4. Places `1` for the matching category.
5. Places `0` for all other categories.

The original `City` column is replaced by the encoded columns.

## 5.1 One-Hot Encoding with `drop_first=True`

To remove one dummy column:

```python
pd.get_dummies(
    city_df,
    columns=['City'],
    drop_first=True,
    dtype=int
)
```

The first category is dropped alphabetically by default in this simple example.

The dropped category becomes the reference category.

In [ ]:
pd.get_dummies(
    city_df,
    columns=['City'],
    drop_first=True,
    dtype=int
)

### Important

`drop_first=True` is useful when we want to avoid redundant dummy columns, especially in linear models.

However, dropping a column is not always mandatory for every algorithm.

For example, many tree-based models can work with all one-hot columns without the same multicollinearity concern.

<div style="border-top:black 2px solid"></div>

# 6. Dummy Variable Trap

## 6.1 What is the Dummy Variable Trap?

The Dummy Variable Trap occurs when we include all dummy variables for a categorical feature along with an intercept in a model that assumes independent predictors.

This creates perfect multicollinearity.

### Example

Suppose a column `Gender` contains only:

```text
Male
Female
```

After One-Hot Encoding:

| Gender_Male | Gender_Female |
|---:|---:|
| 1 | 0 |
| 0 | 1 |
| 1 | 0 |
| 0 | 1 |

For every row:

\[
Gender\_Male + Gender\_Female = 1
\]

Therefore, one column can be perfectly calculated from the other column and the intercept.

This means the columns contain redundant information.

## 6.2 Why does this create a problem?

Suppose a linear regression model is:

\[
y = b_0 + b_1(Gender\_Male) + b_2(Gender\_Female)
\]

Because:

\[
Gender\_Female = 1 - Gender\_Male
\]

the model contains redundant information.

This can make the coefficient estimates unstable or difficult to interpret.

The issue is called **perfect multicollinearity**.

### Important words

- **Multicollinearity** → Predictors are strongly related to one another.
- **Perfect multicollinearity** → One predictor can be exactly represented using other predictors.
- **Dummy Variable Trap** → A common case of perfect multicollinearity caused by including all dummy variables with an intercept.

## 6.3 How do we remove the Dummy Variable Trap?

For a categorical feature with `k` categories:

```text
Create k dummy columns
Remove one dummy column
Keep k - 1 columns
```

Example:

```text
Gender → Male, Female
```

Create:

```text
Gender_Male
Gender_Female
```

Remove one:

```text
Gender_Male
```

Now:

- `Gender_Male = 1` means Male.
- `Gender_Male = 0` means Female.

The remaining category is represented by all zeros.

### Reference category

The removed category is called the **reference category** or **baseline category**.

<div style="border-top:black 2px solid"></div>

# 7. Practical Example using the Customer Dataset

We will use the provided customer dataset.

Columns include:

- `age`
- `gender`
- `review`
- `education`
- `purchased`

For this notebook:

- `gender` is nominal and suitable for One-Hot Encoding.
- `review` and `education` have order and were handled using Ordinal Encoding earlier.
- `purchased` is the target.

In [ ]:
df = pd.read_csv('customer(1).csv')

df.head()

In [ ]:
df.info()

In [ ]:
df['gender'].value_counts()

## 7.1 Encoding the Gender Column

Gender is a categorical feature.

We will convert it into binary columns.

In [ ]:
gender_encoded = pd.get_dummies(
    df[['gender']],
    columns=['gender'],
    dtype=int
)

gender_encoded.head()

### What happened?

The original `gender` column was replaced with separate binary columns.

For example:

```text
gender_Male
gender_Female
```

Each row contains a `1` in the matching category column and `0` in the other category column.

## 7.2 Removing One Dummy Variable

In [ ]:
gender_encoded_drop_first = pd.get_dummies(
    df[['gender']],
    columns=['gender'],
    drop_first=True,
    dtype=int
)

gender_encoded_drop_first.head()

### Interpretation

If the remaining column is `gender_Male`:

- `1` → Male
- `0` → Female, the reference category

We have retained the same information using one fewer column.

<div style="border-top:black 2px solid"></div>

# 8. One-Hot Encoding using Scikit-learn

Scikit-learn provides:

```python
OneHotEncoder
```

It is more suitable for production Machine Learning pipelines because it works naturally with:

- Train-test splits
- Pipelines
- ColumnTransformer
- Unknown categories
- Sparse matrices

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    sparse_output=False,
    dtype=int
)

ohe

In [ ]:
gender_array = ohe.fit_transform(
    df[['gender']]
)

gender_array[:5]

In [ ]:
ohe.categories_

In [ ]:
gender_encoded_sklearn = pd.DataFrame(
    gender_array,
    columns=ohe.get_feature_names_out(['gender']),
    index=df.index
)

gender_encoded_sklearn.head()

### Important methods and attributes

| Method / Attribute | Meaning |
|---|---|
| `fit()` | Learns the categories |
| `transform()` | Converts categories into encoded values |
| `fit_transform()` | Fits and transforms together |
| `categories_` | Shows learned categories |
| `get_feature_names_out()` | Gives names of encoded columns |
| `inverse_transform()` | Converts encoded values back to categories |

## 8.1 OneHotEncoder with `drop='first'`

Scikit-learn can remove the first dummy column directly.

In [ ]:
ohe_drop = OneHotEncoder(
    drop='first',
    sparse_output=False,
    dtype=int
)

gender_drop_array = ohe_drop.fit_transform(
    df[['gender']]
)

pd.DataFrame(
    gender_drop_array,
    columns=ohe_drop.get_feature_names_out(['gender']),
    index=df.index
).head()

## 8.2 Handling Unknown Categories

Production data may contain a category that was not present during training.

Example:

```text
Training data: Male, Female
Future data: Other
```

To avoid an error, use:

```python
handle_unknown='ignore'
```

Unknown categories are represented by zeros across the encoded columns.

In [ ]:
ohe_safe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False,
    dtype=int
)

ohe_safe.fit(df[['gender']])

new_gender = pd.DataFrame({
    'gender': ['Other']
})

ohe_safe.transform(new_gender)

<div style="border-top:black 2px solid"></div>

# 9. OHE using Most Frequent Variables

This phrase can refer to two related practical ideas.

## A. Encoding the most frequent categories

We can inspect the frequency of categories using:

```python
value_counts()
```

For example, if a city appears very frequently, we can study its frequency before encoding.

In [ ]:
df['gender'].value_counts()

## B. Handling rare categories by grouping them

In real datasets, a categorical column may contain hundreds of categories.

Some categories may appear only once or twice.

Creating a separate one-hot column for every rare category can produce:

- Too many columns
- Sparse data
- Increased memory usage
- More complex models

A common strategy is:

1. Find the most frequent categories.
2. Keep those categories separately.
3. Replace all other rare categories with `Other`.
4. Apply One-Hot Encoding.

### Example

```text
Delhi      → Delhi
Mumbai     → Mumbai
Pune       → Pune
Jaipur     → Other
Chennai    → Other
Kochi      → Other
```

This is often called **top-k encoding** or **rare-category grouping**, followed by One-Hot Encoding.

In [ ]:
city_data = pd.DataFrame({
    'City': [
        'Delhi', 'Delhi', 'Delhi',
        'Mumbai', 'Mumbai',
        'Pune',
        'Jaipur', 'Chennai', 'Kochi'
    ]
})

city_data['City'].value_counts()

In [ ]:
top_categories = city_data['City'].value_counts().nlargest(2).index

city_data['City_Grouped'] = city_data['City'].where(
    city_data['City'].isin(top_categories),
    'Other'
)

city_data

In [ ]:
pd.get_dummies(
    city_data,
    columns=['City_Grouped'],
    dtype=int
)

### Why do we group rare categories?

It can help when:

- There are many unique categories.
- Rare categories do not have enough observations.
- We want to control the number of encoded columns.
- The model does not need a separate column for every rare category.

### Important warning

Do not group categories blindly.

A rare category may still be highly important. Use domain knowledge and validation.

<div style="border-top:black 2px solid"></div>

# 10. One-Hot Encoding vs Ordinal Encoding

| One-Hot Encoding | Ordinal Encoding |
|---|---|
| Creates separate binary columns | Creates numerical category labels |
| Best for nominal categories | Best for ordered categories |
| Does not create artificial ranking | Preserves meaningful ranking |
| Can increase number of columns | Usually keeps the same number of columns |
| Example: City → Delhi, Mumbai, Pune columns | Review → Poor=0, Average=1, Good=2 |

### Example

#### City

```text
Delhi, Mumbai, Pune
```

Use One-Hot Encoding.

#### Review

```text
Poor < Average < Good
```

Use Ordinal Encoding.

<div style="border-top:black 2px solid"></div>

# 11. Important Practical Rules

1. Use One-Hot Encoding mainly for nominal categorical input features.
2. Do not create arbitrary rankings for cities, colours, or browser types.
3. Remove one dummy column when required to avoid perfect multicollinearity.
4. Use `drop='first'` in Scikit-learn when appropriate.
5. Fit the encoder on training data only.
6. Transform test data using the same fitted encoder.
7. Use `handle_unknown='ignore'` for safer production pipelines.
8. Group rare categories when the number of unique values is very high.
9. Do not encode the target using One-Hot Encoding unless the model/problem specifically requires it.
10. Check the number of output columns after encoding.

# 12. Final Revision

## One-Hot Encoding

Converts each category into a separate binary column containing `0` and `1`.

## Dummy Variable

A binary column representing whether a row belongs to a category.

## Dummy Variable Trap

Occurs when all dummy columns are included along with an intercept, creating perfect multicollinearity.

## How to remove the trap?

For `k` categories:

```text
Create k dummy variables
Remove 1 dummy variable
Keep k - 1 variables
```

The removed category becomes the reference category.

## Pandas

```python
pd.get_dummies(
    df,
    columns=['gender'],
    drop_first=True,
    dtype=int
)
```

## Scikit-learn

```python
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop='first',
    handle_unknown='ignore',
    sparse_output=False
)
```

## Rare categories

Keep the most frequent categories and group the remaining categories as `Other` before applying OHE.

### Final memory trick

```text
Nominal category → One-Hot Encoding
Ordered category → Ordinal Encoding
Target labels → Label Encoding
All dummies + intercept → Dummy Variable Trap
k categories → usually k - 1 columns when dropping one
```

> **One-Hot Encoding converts categories into independent binary indicators without imposing an artificial ranking.**